In [1]:
import numpy as np
import pandas as pd
from scipy.io import loadmat

import pandas as pd
import time
import os
import sys
import zarr
import napari 
import dask.array as da 

pythonPackagePath = os.path.abspath(r'C:\Users\Kenny\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)
# from parallel import Detector
from gaussian_visualization import visualize_3D_gaussians

In [2]:
base_dir =  r'\\10.158.28.194\ActiveNAS\Kenny\SiRActinData'

base_dir =  r'Z:\Abhi\LLSM_Analysis'
# Define the file directory and name
input_file_directory = '100nM_SiRactin_analysis/'
# zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'

# zarr_full_path = os.path.join(base_dir, zarr_file_directory)

In [3]:
# Load the tracks
if import_trackability:
    path_to_tracks = os.path.join(base_dir, input_file_directory_trackability) + 'trackInROI/Channel_1_tracking_result.mat' 
    path_to_trackability = os.path.join(base_dir, input_file_directory_trackability) + 'trackInROI/Channel_1_tracking_result_Trackability.mat'  
else:
    path_to_tracks = os.path.join(base_dir, input_file_directory) + 'Channel_1_tracking_result.mat'


tracks_dict = loadmat(path_to_tracks)
if import_trackability:
    trackability_dict = loadmat(path_to_trackability)

In [4]:
# Get the keys of the dictionary

# tracks_dict.keys()
# if import_trackability:
    # trackability_dict.keys()

In [4]:
# Get the key for your detections
key = 'tracksFinal'  # Replace with your actual key name
if import_trackability:
    key_trackability = 'trackabilityData'  # Replace with your actual key name

In [5]:
def convert_tracks_to_dataframe(tracks_dict, key):
    """
    Convert MATLAB tracking data to a pandas DataFrame.
    
    Parameters:
    tracks_dict : dict
        Dictionary containing the MATLAB data (loaded using scipy.io.loadmat)
    key : str
        Key for the specific tracking data in the dictionary
    
    Returns:
    pd.DataFrame: DataFrame with columns [frame, mu_x, mu_y, mu_z, track_id]
    """
    import numpy as np
    import pandas as pd
    import scipy.io as sio
    
    # # Load the MATLAB file
    # mat_data = sio.loadmat(filepath, squeeze_me=False, struct_as_record=False)
    
    # Extract the tracks structure
    tracks = tracks_dict[key].flatten()
    
    # Create an empty list to store all track data
    all_tracks = []
    regular_tracks = []
    split_merge_tracks = []
    
    # Process each track
    for track_id in range(len(tracks)):

        # Handle numpy.void objects by accessing elements with field names
        track = tracks[track_id]
        # Access fields using dictionary-like indexing for numpy.void objects
        track_coords = track['tracksCoordAmpCG']
        seq_events = track['seqOfEvents']
        
        # Handle different shapes of seqOfEvents
        if seq_events.size == 0:
            continue  # Skip empty tracks
            
        # Reshape if necessary to ensure consistent format
        if len(seq_events.shape) == 1:
            seq_events = seq_events.reshape(1, -1)

        if np.isnan(seq_events[:, -1]).all():

            track_coords = track_coords[0]

            # save the track as a regular track
            regular_tracks.append(track_id)

            # Find start and end frames
            start_frame = int(seq_events[0, 0])
            end_frame = int(seq_events[-1, 0])

            # Determine the total number of frames from the size of tracksCoordAmpCG
            # Each frame has 8 columns [x y z a dx dy dz da]
            num_cols = track_coords.shape[0]
            num_frames = num_cols // 8

            # For each frame in the track's lifespan
            for frame_idx in range(num_frames):
                frame_number = start_frame + frame_idx
                # Extract x, y, z coordinates for current frame
                col_idx = frame_idx * 8

                x = track_coords[col_idx]
                y = track_coords[col_idx + 1]
                z = track_coords[col_idx + 2]
                amplitude = track_coords[col_idx + 3]

                track_data = {
                    'frame': frame_number,
                    'mu_x': round(x) if not np.isnan(x) else None,
                    'mu_y': round(y) if not np.isnan(y) else None,
                    'mu_z': round(z) if not np.isnan(z) else None,
                    'amplitude': amplitude,
                    'track_id': track_id  
                }

                all_tracks.append(track_data)

        else:
            # save the track as a split/merge track
            split_merge_tracks.append(track_id)

            # Determine if the track is split or merged (this seems a bit rudimentary, probably could be better)
            # Abhishek Raghunathan, 04/14/25
            if not np.isnan(seq_events[1, -1]): # This is assuming that we have only a single split event, it will fail otherwise.
                # Also that all split events are position 1 in seq_events
                track_flag = 'split'
                # print(f'Track {track_id} is split.')

            if not np.isnan(seq_events[2, -1]): # This is assuming that we have only a single merge event, it will fail otherwise.
                # Also that all merge events are position 2 in seq_events
                track_flag = 'merge'
                # print(f'Track {track_id} is merged.')
            

            # Process each segment in the track
            segments = np.unique(seq_events[:, 2]).astype(int)
            segments_min = np.min(segments) # Assuming the lowest segment ID is the first one (might not be true).
            start_frame_original = [] #To store the original start frame
            
            for segment_id in segments:
                # Find events related to this segment
                segment_events = seq_events[seq_events[:, 2] == segment_id]
                
                # Get start and end frames for this segment
                start_events = segment_events[segment_events[:, 1] == 1]
                end_events = segment_events[segment_events[:, 1] == 2]

                if segment_id == segments_min: # Assuming the lowest segment ID has the track which started first (lower value of first frame). CHECK THIS.
                    start_frame_original.append(int(start_events[0,0])) # This is the original start frame for the split or merge track
                    # print(start_frame_original)
                
                if start_events.size > 0 and end_events.size > 0:
                    start_frame = int(start_events[0, 0])
                    end_frame = int(end_events[0, 0])
                    
                    # Get row index for this segment (0-indexed)
                    segment_idx = segment_id - 1

                    # Get the row data for this segment
                    if segment_idx < len(track_coords):
                        segment_data = track_coords[segment_idx]
                        
                        # Calculate number of frames in this segment
                        segment_frames = end_frame - start_frame + 1
                        
                        # Process each frame in this segment
                        for frame_offset in range(segment_frames):
                            frame_number = start_frame + frame_offset

                            if (track_flag == 'split') or (track_flag == 'merge'): # This flag is unnecessary here, have it for legacy reasons.
                                col_idx = (start_frame + frame_offset - start_frame_original[0]) * 8 # This will handle cases where start_frame_original is not 1
                            else:
                                col_idx = frame_offset * 8
                            
                            # if track_id == 4370:
                            #     print(col_idx)
                            
                            
                            # Check if indices are within bounds
                            if col_idx + 3 < len(segment_data):
                                x = segment_data[col_idx]
                                y = segment_data[col_idx + 1]
                                z = segment_data[col_idx + 2]
                                amplitude = segment_data[col_idx + 3]
                                
                                track_data = {
                                    'frame': frame_number,
                                    'mu_x': round(x) if not np.isnan(x) else None,
                                    'mu_y': round(y) if not np.isnan(y) else None,
                                    'mu_z': round(z) if not np.isnan(z) else None,
                                    'amplitude': amplitude,
                                    'track_id': track_id,
                                    'segment_id': segment_id
                                }
                                all_tracks.append(track_data)
        
    
    track_df = pd.DataFrame(all_tracks)
    
    # Sort by track_id and frame
    track_df = track_df.sort_values(['track_id', 'frame'])

    return track_df, regular_tracks, split_merge_tracks

In [6]:
df, regular_tracks, split_merge_tracks = convert_tracks_to_dataframe(tracks_dict, key)

In [7]:
# # Diagnostic code to find the right path to trackability data
# def find_trackability_path(trackability_dict, key_trackability):
#     """
#     Navigate through the MATLAB structure to find the actual trackability data
#     """
#     print("=== FINDING TRACKABILITY DATA PATH ===\n")
    
#     trackability_data = trackability_dict[key_trackability]
#     print(f"1. trackability_data shape: {trackability_data.shape}")
#     print(f"   trackability_data type: {type(trackability_data)}")
    
#     # Level 1: trackability_data[0]
#     level1 = trackability_data[0]
#     print(f"\n2. trackability_data[0] shape: {level1.shape}")
#     print(f"   trackability_data[0] type: {type(level1)}")
    
#     # Level 2: trackability_data[0][0]  
#     level2 = level1[0]
#     print(f"\n3. trackability_data[0][0] shape: {level2.shape}")
#     print(f"   trackability_data[0][0] type: {type(level2)}")
#     print(f"   trackability_data[0][0] dtype: {level2.dtype}")
    
#     # Check if it has named fields
#     if hasattr(level2.dtype, 'names') and level2.dtype.names:
#         print(f"   Field names: {level2.dtype.names}")
        
#         # Look for segTrackability field
#         if 'segTrackability' in level2.dtype.names:
#             seg_track = level2['segTrackability']
#             print(f"\n4. Found segTrackability field!")
#             print(f"   segTrackability shape: {seg_track.shape}")
#             print(f"   segTrackability type: {type(seg_track)}")
            
#             # Try to access the actual data
#             if seg_track.shape == (1,):
#                 actual_data = seg_track[0]
#                 print(f"\n5. segTrackability[0] shape: {actual_data.shape}")
#                 print(f"   segTrackability[0] type: {type(actual_data)}")
                
#                 # Show first few entries
#                 print(f"\n6. First few trackability arrays:")
#                 for i in range(min(3, len(actual_data))):
#                     print(f"   Track {i}: {actual_data[i].flatten()[:10]}...")
                    
#                 return actual_data
#             else:
#                 print(f"\n5. segTrackability data directly accessible")
#                 return seg_track
#         else:
#             print("   segTrackability field not found in named fields")
            
#             # Try to access the data directly
#             if level2.shape == (1,):
#                 level3 = level2[0]
#                 print(f"\n4. trackability_data[0][0][0] shape: {level3.shape}")
#                 print(f"   trackability_data[0][0][0] type: {type(level3)}")
                
#                 # If it's an array, try to access elements
#                 if len(level3) > 0:
#                     print(f"\n5. First element: {level3[0]}")
#                     if hasattr(level3[0], 'shape'):
#                         print(f"   First element shape: {level3[0].shape}")
                        
#                 return level3
#     else:
#         print("   No named fields, trying direct access")
#         if level2.shape == (1,):
#             level3 = level2[0]
#             print(f"\n4. trackability_data[0][0][0] shape: {level3.shape}")
#             print(f"   trackability_data[0][0][0] type: {type(level3)}")
#             return level3
#         else:
#             return level2

# # Run the diagnostic
# actual_trackability_data = find_trackability_path(trackability_dict, key_trackability)

In [ ]:
len(split_merge_tracks)
# split_merge_tracks

44

: 

In [12]:
# df
# df[(df['track_id'] == 37) & (df['segment_id'] == 1.0)]

In [13]:
#### Testing out code to identify DNM2 associated AP2 spots ####
dnm2_dectections = 'controlOS_analysis/ch2/detection'
path_to_dnm2_detections = os.path.join(base_dir, dnm2_dectections) + '/channel_1_detections.pkl'
dnm2_df = pd.read_pickle(path_to_dnm2_detections).reset_index()

In [14]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from tqdm import tqdm

def find_dnm2_at_ap2_positions(df, dnm2_df, search_radius=2, min_consecutive=3, verbose=True):
    """
    Identify DNM2 spots near AP2 track positions using efficient spatial search.
    
    For each AP2 spot, searches for DNM2 detections within a cubic region of
    side length (2*search_radius + 1) voxels centered on the AP2 position.
    Example: search_radius=2 creates a 5×5×5 voxel cube.
    
    The search uses Chebyshev distance (L∞ norm):
        max(|dx|, |dy|, |dz|) ≤ search_radius
    
    This ensures the search region is exactly ±search_radius voxels in each
    dimension from the AP2 center.
    
    Track-level classification: A track is considered DNM2-positive if it has
    at least min_consecutive consecutive frames with DNM2 detections.
    
    Parameters:
    -----------
    df : pandas DataFrame
        AP2 track dataframe with columns ['mu_x', 'mu_y', 'mu_z', 'frame', 'track_id', ...]
    dnm2_df : pandas DataFrame
        DNM2 detection dataframe with columns ['mu_x', 'mu_y', 'mu_z', 'frame', ...]
    search_radius : int, default=2
        Search radius in voxels. Creates cube of side (2*radius + 1) voxels.
        Example: radius=2 → 5×5×5 cube (center ± 2 voxels)
    min_consecutive : int, default=3
        Minimum number of consecutive DNM2-positive frames required for track
        to be classified as DNM2-positive
    verbose : bool, default=True
        If True, print detection statistics
        
    Returns:
    --------
    pandas DataFrame
        Copy of df with added columns:
        - 'dnm2_positive': bool, True if DNM2 spot found within search radius
        - 'dnm2_mu_x': float or 'two_spots', x-coordinate of nearest DNM2 spot
        - 'dnm2_mu_y': float or 'two_spots', y-coordinate of nearest DNM2 spot
        - 'dnm2_mu_z': float or 'two_spots', z-coordinate of nearest DNM2 spot
        - 'dnm2_positive_track': bool, True if track has ≥min_consecutive consecutive DNM2+ frames
    """
    
    # Create copy to avoid modifying original dataframe
    result_df = df.copy()
    
    # Initialize output columns with object dtype to accommodate both floats and strings
    # This prevents FutureWarning when mixing 'two_spots' string with float values
    result_df['dnm2_positive'] = False
    result_df['dnm2_mu_x'] = pd.Series(dtype='object')  # object dtype allows mixed types
    result_df['dnm2_mu_y'] = pd.Series(dtype='object')
    result_df['dnm2_mu_z'] = pd.Series(dtype='object')
    
    # Fill with None initially (will show as NaN for numeric operations)
    result_df['dnm2_mu_x'] = None
    result_df['dnm2_mu_y'] = None
    result_df['dnm2_mu_z'] = None
    
    # Counters for statistics
    n_successful = 0      # AP2 spots with exactly 1 closest DNM2 spot
    n_equidistant = 0     # AP2 spots with ≥2 equidistant DNM2 spots
    
    # Group both dataframes by frame for efficient per-frame processing
    # This avoids searching across irrelevant frames
    ap2_grouped = result_df.groupby('frame')
    dnm2_grouped = dnm2_df.groupby('frame')
    
    # Get set of frames present in DNM2 data for quick lookup
    dnm2_frames = set(dnm2_df['frame'].unique())
    
    # Get sorted list of frames for progress bar
    frames_to_process = sorted(ap2_grouped.groups.keys())
    
    # Process each frame independently with progress bar
    for frame_idx in tqdm(frames_to_process, desc="Processing frames", disable=not verbose):
        
        ap2_frame_df = ap2_grouped.get_group(frame_idx)
        
        # Skip frames with no DNM2 detections (optimization)
        if frame_idx not in dnm2_frames:
            continue
        
        # Get DNM2 detections for this frame
        dnm2_frame_df = dnm2_grouped.get_group(frame_idx)
        
        # Extract coordinates as numpy arrays for vectorized operations
        # Shape: (n_ap2_spots, 3)
        ap2_coords = ap2_frame_df[['mu_x', 'mu_y', 'mu_z']].values
        
        # Shape: (n_dnm2_spots, 3)
        dnm2_coords = dnm2_frame_df[['mu_x', 'mu_y', 'mu_z']].values
        
        # Build KD-tree for fast spatial queries
        # KD-tree enables O(log N) nearest neighbor search vs O(N) brute force
        dnm2_tree = cKDTree(dnm2_coords)
        
        # Query for all DNM2 spots within search_radius of each AP2 spot
        # query_ball_point returns list of indices for each query point
        # Uses Chebyshev distance (L∞ norm) to create cubic search region:
        #   max(|dx|, |dy|, |dz|) ≤ search_radius
        # This creates a cube of side (2*search_radius + 1) voxels
        neighbors_lists = dnm2_tree.query_ball_point(
            ap2_coords,
            r=search_radius,
            p=np.inf  # Chebyshev distance for cubic region
        )
        
        # Process each AP2 spot in this frame
        for i, (ap2_idx, neighbor_indices) in enumerate(zip(ap2_frame_df.index, neighbors_lists)):
            
            # No DNM2 spots found within search radius
            if len(neighbor_indices) == 0:
                # dnm2_positive already False, coordinates already None
                continue
            
            # Extract the AP2 coordinates for distance calculation
            ap2_coord = ap2_coords[i]
            
            # Get coordinates of all neighboring DNM2 spots
            neighbor_coords = dnm2_coords[neighbor_indices]
            
            # Calculate Euclidean distances to all neighbors
            # This is used to break ties when multiple DNM2 spots are within the cube
            # Shape: (n_neighbors,)
            distances = np.sqrt(np.sum((neighbor_coords - ap2_coord)**2, axis=1))
            
            # Find minimum distance
            min_distance = np.min(distances)
            
            # Find all spots at minimum distance (handles ties)
            # Using small tolerance for floating point comparison
            closest_mask = np.abs(distances - min_distance) < 1e-10
            closest_indices = np.where(closest_mask)[0]
            
            # Handle equidistant case (≥2 spots at same distance)
            if len(closest_indices) > 1:
                n_equidistant += 1
                result_df.loc[ap2_idx, 'dnm2_positive'] = True
                result_df.loc[ap2_idx, 'dnm2_mu_x'] = 'two_spots'
                result_df.loc[ap2_idx, 'dnm2_mu_y'] = 'two_spots'
                result_df.loc[ap2_idx, 'dnm2_mu_z'] = 'two_spots'
            else:
                # Single closest DNM2 spot found
                n_successful += 1
                
                closest_local_idx = closest_indices[0]
                closest_global_idx = neighbor_indices[closest_local_idx]
                
                # Retrieve coordinates from original DNM2 dataframe
                closest_dnm2_row = dnm2_frame_df.iloc[closest_global_idx]
                
                # Update result dataframe
                result_df.loc[ap2_idx, 'dnm2_positive'] = True
                result_df.loc[ap2_idx, 'dnm2_mu_x'] = closest_dnm2_row['mu_x']
                result_df.loc[ap2_idx, 'dnm2_mu_y'] = closest_dnm2_row['mu_y']
                result_df.loc[ap2_idx, 'dnm2_mu_z'] = closest_dnm2_row['mu_z']
    
    # Calculate number of DNM2-negative AP2 spots
    n_negative = len(result_df) - n_successful - n_equidistant
    
    # =========================================================================
    # Track-level classification: Find tracks with ≥min_consecutive consecutive DNM2+ frames
    # =========================================================================
    
    # Initialize track-level column
    result_df['dnm2_positive_track'] = False
    
    # Group by track_id for per-track analysis
    track_groups = result_df.groupby('track_id')
    
    # Set to store track IDs that are DNM2-positive
    positive_track_ids = set()
    
    # Process each track with progress bar
    for track_id, track_df in tqdm(track_groups, desc="Classifying tracks", disable=not verbose):
        # Sort by frame to ensure consecutive frame detection
        track_df_sorted = track_df.sort_values('frame')
        
        # Get boolean array of DNM2-positive frames
        dnm2_status = track_df_sorted['dnm2_positive'].values
        
        # Find maximum consecutive run of True values
        # Algorithm: iterate through array tracking current run length
        max_consecutive = 0
        current_consecutive = 0
        
        for is_positive in dnm2_status:
            if is_positive:
                current_consecutive += 1
                max_consecutive = max(max_consecutive, current_consecutive)
            else:
                current_consecutive = 0
        
        # Check if track meets threshold for being DNM2-positive
        if max_consecutive >= min_consecutive:
            positive_track_ids.add(track_id)
    
    # Update dnm2_positive_track column for all rows of positive tracks
    result_df.loc[result_df['track_id'].isin(positive_track_ids), 'dnm2_positive_track'] = True
    
    # Calculate track-level statistics
    n_total_tracks = result_df['track_id'].nunique()
    n_positive_tracks = len(positive_track_ids)
    pct_positive_tracks = 100 * n_positive_tracks / n_total_tracks if n_total_tracks > 0 else 0
    
    # Print comprehensive statistics if requested
    if verbose:
        print(f"Number of AP2 spots with successful DNM2 detection: {n_successful}")
        print(f"Number of DNM2-negative AP2 spots: {n_negative}")
        print(f"Number of AP2 spots with equidistant DNM2 detections: {n_equidistant}")
        print(f"Percentage of DNM2-positive tracks: {pct_positive_tracks:.1f}%")
    
    return result_df


# Example usage:
df_with_dnm2 = find_dnm2_at_ap2_positions(df, dnm2_df, search_radius=2, min_consecutive=3, verbose=True)

Classifying tracks: 100%|██████████| 88089/88089 [00:46<00:00, 1896.31it/s]


Number of AP2 spots with successful DNM2 detection: 353075
Number of DNM2-negative AP2 spots: 372345
Number of AP2 spots with equidistant DNM2 detections: 2422
Percentage of DNM2-positive tracks: 28.7%


In [15]:
# # Save df in path_to_tracks as a pickle file
output_path = path_to_tracks.replace('.mat', '.pkl')
print(f"Saving DataFrame to {output_path}")
df_with_dnm2.to_pickle(output_path)

Saving DataFrame to Z:\Abhi\LLSM_Analysis\controlOS_analysis/tracks/Channel_1_tracking_result.pkl


In [16]:
####This code supports the idea that the last track segment should have trackability score of NaN
#### rather than the first segment.
if import_trackability:
    # for each track_id in df, count the number of NaN values in the segTrackability column
    nan_counts = df.groupby('track_id')['segTrackability'].apply(lambda x: x.isna().sum())
    # Create a DataFrame from the counts
    nan_counts_df = nan_counts.reset_index()
    nan_counts_df.columns = ['track_id', 'nan_count']
    set(nan_counts_df['nan_count'])  # This will show the unique counts of NaN values per track_id
    # Find track ids with more than 1 NaN value in segTrackability
    tracks_with_multiple_nans = nan_counts_df[nan_counts_df['nan_count'] > 1]['track_id'].tolist()
    print(f"Tracks with multiple NaN values in segTrackability: {tracks_with_multiple_nans}")
# df[df['track_id'] == 5]

In [17]:
# tracks = tracks_dict[key].flatten()
# track = tracks[37]
# track_coords = track['tracksCoordAmpCG']
# seq_events = track['seqOfEvents']
# seq_events

In [18]:
# track_coords[0]

In [19]:
# track_df = df[df['track_id'] == 33789]
# #sort based on segment id values
# track_df.sort_values(['segment_id', 'frame'])

In [20]:
# # get non NaN values in track_coords[1]
# track_coords[1][~np.isnan(track_coords[1])]

In [21]:
# # Subset rows of df with NaNs in any of mu_x, mu_y, mu_z
# df_nan = df[df[['mu_x', 'mu_y', 'mu_z']].isnull().any(axis=1)]
# # Get the track IDs of these rows
# nan_track_ids = df_nan['track_id'].unique()
# # Count the number of times each track ID appears in df_nan
# track_id_counts = df_nan['track_id'].value_counts()
# # Filter track IDs that appear more than once
# track_id_counts = track_id_counts[track_id_counts > 1]

